# YOLOv11 PPE Engine - Roboflow Dataset Training (Colab)

This notebook trains on the Roboflow Medical PPE dataset.
Classes: **Coverall, Gloves, Goggles, Mask**

### Step 1: Mount Google Drive
Upload `roboflow.zip` to the root of your Google Drive before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Extract Dataset

In [ ]:
!unzip -q /content/drive/MyDrive/roboflow.zip -d /content/roboflow_dataset

### Step 3: Install Ultralytics

In [ ]:
!pip install -q ultralytics

### Step 4: Fix data.yaml paths for Colab

In [ ]:
import yaml
import os

# Find the data.yaml (it could be at root or one level deep)
yaml_path = '/content/roboflow_dataset/data.yaml'
if not os.path.exists(yaml_path):
    # Search one level deep
    for d in os.listdir('/content/roboflow_dataset'):
        candidate = os.path.join('/content/roboflow_dataset', d, 'data.yaml')
        if os.path.exists(candidate):
            yaml_path = candidate
            break

print(f'Found data.yaml at: {yaml_path}')
base_dir = os.path.dirname(yaml_path)

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = base_dir
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('data.yaml updated!')
print(f'Classes: {data["names"]}')
print(f'Num classes: {data["nc"]}')

### Step 5: Train!
Using YOLO11s (small) since Colab T4 GPU is much faster than your laptop.
Results are saved directly back to Google Drive.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # Using 'small' model since Colab GPU is powerful enough

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,          # Full resolution since Colab GPU can handle it
    batch=32,
    project='/content/drive/MyDrive/runs/detect',
    name='ppe_roboflow_colab',
    exist_ok=True,
    patience=20
)

### Step 6: View Results

In [ ]:
from IPython.display import Image, display

results_dir = '/content/drive/MyDrive/runs/detect/ppe_roboflow_colab'

print('--- Confusion Matrix ---')
display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

print('\n--- Training Results ---')
display(Image(filename=f'{results_dir}/results.png', width=800))

print('\n--- Validation Predictions ---')
display(Image(filename=f'{results_dir}/val_batch0_pred.jpg', width=800))